# Build the feature-engineered dataset: `papers_combined.parquet` -> `papers_fe.parquet`

One job: assemble every already-validated, row-local feature block (Tier 1b lexical
features, the three winning cached embeddings, brief-similarity, admissible raw
metadata) into one table for `sf_*_fold_pipeline.ipynb` to split. This notebook does
not model, score, split, or tune anything — the governing rule throughout is that a
column only belongs here if its value would be the same regardless of which rows land
in a training fold. PCA, scalers, TF-IDF vocabularies, imputers, and the Tier 2
applied-vs-foundational anchor direction are all fold-dependent and stay out.

## 1. Config and schema check

`fold_pipeline_utils.validate_schema` reads its required-column list out of a
fold-pipeline-shaped `CONFIG` dict (`target_col`, `embedding_col`,
`numeric_feature_cols`, ...) built for the `sf_*_fold_pipeline.ipynb` notebooks — not a
generic "does this table have the columns I need" checker. Reusing it here would mean
building a fake CONFIG just to satisfy its shape, so this notebook does its own
explicit check instead: same "fail loudly here, not three cells later" goal, right-sized
tool.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../../scripts")
from fold_pipeline_utils import derive_first_author
from lexical_features import build_lexical_features, derangements

DATA_PATH = Path("../../data/processed/papers_combined.parquet")
CACHE_DIR = Path("../../data/processed/embeddings_cache")
OUTPUT_PATH = Path("../../data/processed/papers_fe.parquet")

# model_name -> (cache file stem, output column prefix)
EMBEDDING_MODELS = {
    "infgrad/Jasper-Token-Compression-600M": "jasper",
    "Qwen/Qwen3-Embedding-4B": "qwen4b",
    "qwen/qwen3-embedding-8b": "qwen8b",
}

BUILD_CONTROL = False  # True re-runs Tier 1b against deliberately wrong briefs (lexctl_*)

REQUIRED_COLS = [
    "paper_id", "use_case_key", "triage_label", "authors", "title", "abstract",
    "doi", "year", "citation_count", "has_abstract", "exported_at",
    "objective", "problem_statement", "terms_must_include", "terms_nice_to_have",
    "terms_exclude", "domain_industry", "domain_application", "domain_technology_focus",
]

In [2]:
df_raw = pd.read_parquet(DATA_PATH)

missing = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing:
    raise ValueError(
        f"papers_combined.parquet is missing column(s) {missing} this notebook depends "
        f"on - update REQUIRED_COLS or the upstream export before continuing. Columns "
        f"present: {sorted(df_raw.columns)}"
    )
print(f"Loaded {DATA_PATH}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols, schema OK.")

Loaded ../../data/processed/papers_combined.parquet: 2873 rows x 50 cols, schema OK.


## 2. Load and filter

Keep only `triage_label in {positive, negative}` — `pass` and never-triaged rows carry
no target and are dropped, not silently coerced. `y` is a pure row-local function of
`triage_label`, so it's safe to materialise here; the fold pipelines can keep deriving
their own copy from `triage_label` too, this doesn't conflict with that.

In [3]:
n_total = len(df_raw)
label_counts = df_raw["triage_label"].value_counts(dropna=False)
print("triage_label counts (full corpus):")
print(label_counts)

pass_rows = df_raw[df_raw["triage_label"] == "pass"]
pass_missing_abstract = (~pass_rows["has_abstract"]).sum()
print(
    f"\n'pass' rows: {len(pass_rows)}, of which {pass_missing_abstract} "
    f"({pass_missing_abstract / len(pass_rows):.1%}) are missing an abstract. "
    f"Dropped, not silently fillna'd — 'pass' and 'never triaged' both encode "
    f"'no target label', not 'negative'."
)

df = df_raw[df_raw["triage_label"].isin(["positive", "negative"])].reset_index(drop=True)
df["y"] = (df["triage_label"] == "positive").astype(bool)

print(f"\nRetained {len(df)} / {n_total} rows ({len(df) / n_total:.1%}).")
print("\nPer-use-case prevalence (share positive):")
print(df.groupby("use_case_key")["y"].mean().sort_values(ascending=False))

triage_label counts (full corpus):
triage_label
positive    1067
negative     785
pass         543
NaN          478
Name: count, dtype: int64

'pass' rows: 543, of which 460 (84.7%) are missing an abstract. Dropped, not silently fillna'd — 'pass' and 'never triaged' both encode 'no target label', not 'negative'.

Retained 1852 / 2873 rows (64.5%).

Per-use-case prevalence (share positive):
use_case_key
solar_leo           0.766667
ner                 0.714744
cement_binders      0.648855
tech_forecasting    0.594697
carbon_capture      0.494949
soil_microbiome     0.263305
Name: y, dtype: float64


## 3. Dedupe

Scoped to **within `use_case_key`**, not globally. A DOI or near-duplicate title
repeated *across* two different use cases is the same real paper independently labelled
for two different briefs — a legitimate (brief, paper) pair each time, not an accidental
duplicate row. This project's own Tier 1b finding is that relevance is a property of the
(brief, paper) pair, not the paper alone, so deduping globally would silently destroy
real, independently-labelled rows. Deduping within a use case guards against the real
risk (the same paper harvested twice from two source APIs into one use case's pool)
without discarding cross-use-case signal.

In [4]:
def normalise_title(t):
    return " ".join(str(t).lower().split())


df["_norm_title"] = df["title"].map(normalise_title)
has_doi = df["doi"].fillna("").str.strip() != ""

within_doi = df.duplicated(subset=["use_case_key", "doi"], keep="first") & has_doi
within_title = df.duplicated(subset=["use_case_key", "_norm_title"], keep="first")
cross_doi_informational = df.duplicated(subset=["doi"], keep=False) & has_doi & ~df.duplicated(
    subset=["use_case_key", "doi"], keep=False
)
cross_title_informational = df.duplicated(subset=["_norm_title"], keep=False) & ~df.duplicated(
    subset=["use_case_key", "_norm_title"], keep=False
)

drop_mask = within_doi | within_title
print(f"Within-use_case_key duplicate DOI rows dropped: {within_doi.sum()}")
print(f"Within-use_case_key near-duplicate title rows dropped: {within_title.sum()}")
print(f"Total rows dropped: {drop_mask.sum()}")
print(
    f"\n(Informational only, NOT dropped) cross-use_case_key DOI matches: "
    f"{cross_doi_informational.sum()} rows, title matches: {cross_title_informational.sum()} rows "
    f"- same real paper independently labelled under a different brief, kept."
)

df = df[~drop_mask].drop(columns="_norm_title").reset_index(drop=True)
n_after_dedupe = len(df)
print(f"\nRows after dedupe: {n_after_dedupe}")

Within-use_case_key duplicate DOI rows dropped: 0
Within-use_case_key near-duplicate title rows dropped: 4
Total rows dropped: 4

(Informational only, NOT dropped) cross-use_case_key DOI matches: 0 rows, title matches: 0 rows - same real paper independently labelled under a different brief, kept.

Rows after dedupe: 1848


## 4. Group key

`derive_first_author` (`scripts/fold_pipeline_utils.py`) — reused rather than
re-derived, so the fold pipelines' grouping and this notebook's can never drift apart.
Rows with no parseable author each get a unique `no_author_<index>` singleton rather
than colliding into one shared "unknown" group.

In [5]:
df["first_author"] = derive_first_author(df["authors"])
n_singleton = df["first_author"].str.startswith("no_author_").sum()
print(f"first_author derived. {n_singleton} rows had no parseable author (unique singleton each).")
print(f"{df['first_author'].nunique()} distinct first-author groups across {len(df)} rows.")

first_author derived. 25 rows had no parseable author (unique singleton each).
1732 distinct first-author groups across 1848 rows.


## 5. Tier 1b lexical block

`build_lexical_features` (`scripts/lexical_features.py`), real briefs (default
`brief_map=None`). 22 columns, nothing fitted on labels — safe to materialise as-is.
Prefixed `lex_` here since the function itself returns unprefixed names.

In [6]:
lexical = build_lexical_features(df).add_prefix("lex_")
assert lexical.shape[1] == 22, f"expected 22 lexical columns, got {lexical.shape[1]}"
print(f"Built {lexical.shape[1]} lex_* columns for {lexical.shape[0]} rows.")
df = pd.concat([df, lexical], axis=1)

Built 22 lex_* columns for 1848 rows.


## 6. Tier 1b control columns (optional, off by default)

`BUILD_CONTROL = True` re-runs the falsification control (deliberately wrong briefs, via
`derangements`) and emits it as `lexctl_*`. Off for production output — this is what
makes the block's "real briefs beat wrong ones" claim re-checkable on demand rather than
resting on `reports/wf_tier1b_lexical_control.md` alone.

In [7]:
if BUILD_CONTROL:
    wrong_brief_map = derangements(sorted(df["use_case_key"].unique()), seed=0)
    control = build_lexical_features(df, brief_map=wrong_brief_map).add_prefix("lexctl_")
    df = pd.concat([df, control], axis=1)
    print(f"Built {control.shape[1]} lexctl_* columns (wrong-brief control).")
else:
    print("BUILD_CONTROL is False - skipping the falsification-control columns.")

BUILD_CONTROL is False - skipping the falsification-control columns.


## 7. Embeddings

Jasper / Qwen3-4B / Qwen3-8B paper vectors, joined on `paper_id`, stored **separately**
(`emb_jasper_*`, `emb_qwen4b_*`, `emb_qwen8b_*`) — concatenation is a modelling choice
that belongs in a fold pipeline, and PCA measurably hurts within-silo, so neither
happens here. Raises if a `paper_id` is missing from a cache rather than filling —
today every cache covers exactly this notebook's labelled row set (0 unmatched either
direction), so this should never trigger; it's a tripwire against future drift.

In [8]:
def safe_name(model_name):
    return model_name.replace("/", "__").replace(":", "_")


embedding_widths = {}
for model_name, prefix in EMBEDDING_MODELS.items():
    cache_path = CACHE_DIR / f"{safe_name(model_name)}_papers.parquet"
    cached = pd.read_parquet(cache_path)

    missing_ids = set(df["paper_id"]) - set(cached["paper_id"])
    if missing_ids:
        raise ValueError(
            f"{model_name}: {len(missing_ids)} paper_id(s) in the FE dataset have no "
            f"cached embedding, e.g. {sorted(missing_ids)[:5]} - re-run the embedding "
            f"cache build before continuing."
        )

    vec_by_id = dict(zip(cached["paper_id"], cached["embedding"]))
    dim = len(next(iter(vec_by_id.values())))
    embedding_widths[prefix] = dim

    matrix = np.stack(df["paper_id"].map(vec_by_id).to_numpy())
    exploded = pd.DataFrame(
        matrix, columns=[f"emb_{prefix}_{i:04d}" for i in range(dim)], index=df.index
    )
    df = pd.concat([df, exploded], axis=1)
    print(f"{model_name}: joined {dim}-dim embedding -> emb_{prefix}_0000..{dim-1:04d}")

print(f"\nTotal embedding columns: {sum(embedding_widths.values())}")

infgrad/Jasper-Token-Compression-600M: joined 2048-dim embedding -> emb_jasper_0000..2047
Qwen/Qwen3-Embedding-4B: joined 2560-dim embedding -> emb_qwen4b_0000..2559


qwen/qwen3-embedding-8b: joined 4096-dim embedding -> emb_qwen8b_0000..4095

Total embedding columns: 8704


## 8. Cosine-to-brief

The zero-label ranker: cosine similarity between each paper's embedding and its own use
case's brief embedding, plus the within-use-case percentile rank (same convention
`lexical_features.py` already uses for `rank_bm25_*`) — materialised as a column, not
reconstructed downstream.

In [9]:
def cosine_sim(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan


for model_name, prefix in EMBEDDING_MODELS.items():
    usecase_path = CACHE_DIR / f"{safe_name(model_name)}_usecases.parquet"
    usecase_df = pd.read_parquet(usecase_path)
    brief_vec = dict(zip(usecase_df["use_case_key"], usecase_df["embedding"]))

    col = f"cos_brief_{prefix}"
    emb_cols = [f"emb_{prefix}_{i:04d}" for i in range(embedding_widths[prefix])]
    df[col] = [
        cosine_sim(row_vec, brief_vec[uc])
        for row_vec, uc in zip(df[emb_cols].to_numpy(), df["use_case_key"])
    ]
    df[f"rank_cos_brief_{prefix}"] = df.groupby("use_case_key")[col].rank(pct=True)
    print(f"{col}: mean={df[col].mean():.3f}, min={df[col].min():.3f}, max={df[col].max():.3f}")

cos_brief_jasper: mean=0.661, min=0.296, max=0.876
cos_brief_qwen4b: mean=0.504, min=0.102, max=0.793
cos_brief_qwen8b: mean=0.514, min=0.132, max=0.814


## 9. Raw admissible metadata

`year`, `paper_age`, `has_abstract`, `n_authors`, `citation_count` — pass-through or
derived raw, NaN preserved, never `fillna(0)`. `paper_age` is derived from each row's own
`exported_at` (a real per-export fact already in the data, not a hardcoded reference
year that goes stale). Admissible here only because today's models are trained one per
use-case silo — a future pooled-across-use-cases model would need to drop these, since
`CONTEXT.md` found a pooled model reads `use_case_key` off metadata like this at high
accuracy.

In [10]:
df["exported_year"] = pd.to_datetime(df["exported_at"]).dt.year
df["paper_age"] = (df["exported_year"] - df["year"]).clip(lower=0)
df = df.drop(columns="exported_year")

def count_authors(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    return float(len(str(value).split(",")))


df["n_authors"] = df["authors"].map(count_authors)

raw_metadata_cols = ["year", "paper_age", "has_abstract", "n_authors", "citation_count"]
print(df[raw_metadata_cols].isna().mean().rename("null_rate"))

year              0.004329
paper_age         0.004329
has_abstract      0.000000
n_authors         0.013528
citation_count    0.109307
Name: null_rate, dtype: float64


## 10. Sanity checks, then write

Assert row count, no all-NaN columns, no infinities, the exclusion-overlap NaN pattern
matches which use cases actually specified exclusion terms, and the three embedding
blocks have the expected widths. One file for now (~60MB, gitignored) — split into a
separate embeddings parquet later only if iterating on the non-embedding columns
becomes slow enough to be annoying.

In [11]:
assert len(df) == n_after_dedupe, f"row count drifted: {len(df)} != {n_after_dedupe}"

all_nan_cols = df.columns[df.isna().all()].tolist()
assert not all_nan_cols, f"all-NaN column(s): {all_nan_cols}"

numeric_df = df.select_dtypes(include=[np.number])
n_inf = np.isinf(numeric_df.to_numpy(dtype=float)).sum()
assert n_inf == 0, f"found {n_inf} infinite values"

has_exclude_terms = df.groupby("use_case_key")["terms_exclude"].first().map(len) > 0
for uc, has_excl in has_exclude_terms.items():
    uc_rows = df["use_case_key"] == uc
    excl_is_nan = df.loc[uc_rows, "lex_overlap_excl_n"].isna().all()
    if has_excl:
        assert not excl_is_nan, f"{uc} has exclusion terms but lex_overlap_excl_n is all-NaN"
    else:
        assert excl_is_nan, f"{uc} has no exclusion terms but lex_overlap_excl_n is not all-NaN"

expected_widths = {"jasper": 2048, "qwen4b": 2560, "qwen8b": 4096}
for prefix, expected_dim in expected_widths.items():
    actual = sum(1 for c in df.columns if c.startswith(f"emb_{prefix}_"))
    assert actual == expected_dim, f"emb_{prefix}_*: expected {expected_dim} cols, got {actual}"

print("All sanity checks passed.")

# papers_combined.parquet carries title/abstract/authors/doi/the brief text fields/
# provenance flags/etc. that earlier steps needed as inputs (dedup, lexical features,
# derive_first_author) but that aren't part of this notebook's output contract - trim to
# exactly the assembled feature groups before writing, rather than passing the rest
# through by accident.
groups = {
    "identifiers/grouping": ["paper_id", "use_case_key", "first_author"],
    "target": ["triage_label", "y"],
    "lexical (Tier 1b)": [c for c in df.columns if c.startswith("lex_")],
    "lexical control": [c for c in df.columns if c.startswith("lexctl_")],
    "embeddings": [c for c in df.columns if c.startswith("emb_")],
    "brief similarity": [c for c in df.columns if c.startswith("cos_brief_") or c.startswith("rank_cos_brief_")],
    "raw metadata": raw_metadata_cols,
}
final_cols = [c for cols in groups.values() for c in cols]
dropped_cols = [c for c in df.columns if c not in final_cols]
print(
    f"\nDropping {len(dropped_cols)} input-only columns not in the output contract "
    f"(still available in papers_combined.parquet): {dropped_cols}"
)
df = df[final_cols]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"\nWrote {OUTPUT_PATH}: {df.shape[0]} rows x {df.shape[1]} cols, {size_mb:.1f} MB")

summary = pd.DataFrame(
    [(name, len(cols), round(df[cols].isna().mean().mean(), 4) if cols else 0.0) for name, cols in groups.items()],
    columns=["group", "n_cols", "mean_null_rate"],
)
print()
print(summary.to_string(index=False))

All sanity checks passed.



Dropping 44 input-only columns not in the output contract (still available in papers_combined.parquet): ['use_case_name', 'source_id', 'title', 'authors', 'venue', 'doi', 'url', 'sources', 'language', 'review_label', 'relevance_score', 'abstract', 'embedding', 'embed_model', 'embed_dim', 'exported_at', 'app_version', 'problem_statement', 'objective', 'usecase_schema_version', 'domain_industry', 'domain_application', 'domain_technology_focus', 'terms_must_include', 'terms_nice_to_have', 'terms_exclude', 'performance_criteria', 'trl_min', 'trl_max', 'constraints_scale', 'constraints_cost', 'decision_must_have', 'decision_nice_to_have', 'decision_exclusions', 'decision_rules', 'notes', 'from_arxiv', 'from_core', 'from_crossref', 'from_europe_pmc', 'from_openalex', 'from_pubmed', 'from_seed', 'from_semantic_scholar']



Wrote ../../data/processed/papers_fe.parquet: 1848 rows x 8742 cols, 101.1 MB

               group  n_cols  mean_null_rate
identifiers/grouping       3          0.0000
              target       2          0.0000
   lexical (Tier 1b)      22          0.0451
     lexical control       0          0.0000
          embeddings    8704          0.0000
    brief similarity       6          0.0000
        raw metadata       5          0.0263
